# New Section

## Anuoluwa Data Cleaning for sentinel code

“Blanks” typically refer to data missing due to: 1) loss to follow-up or 2) data collection not
required for a specific age group, study period, or other reasons. “Unknown” response is
typically coded as 9s, “Not Applicable” is 8s, and “Declined” is 7s. Make sure these 7s, 8s,
and 9s being excluded in the computation of mean, standard deviation, and other statistics.

In [ ]:
def sentinel_codes(series):
    """Return the 7/8/9 sentinel values matching this column's width,
    including decimal sentinels like 88.8 / 99.9 for genuinely
    fractional-valued columns."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return {"declined": pd.NA, "not_applicable": pd.NA, "unknown": pd.NA}

    int_width = max(len(str(abs(int(v)))) for v in s)

    # Correct check: does any value have a non-zero fractional part?
    has_decimal = (s % 1 != 0).any()

    if has_decimal:
        return {
            #"declined": float(f"{'7' * int_width}.7"),
            "not_applicable": float(f"{'8' * int_width}.8"),
            "unknown": float(f"{'9' * int_width}.9"),
        }

    return {
        #"declined": int("7" * int_width),
        "not_applicable": int("8" * int_width),
        "unknown": int("9" * int_width),
    }
  # For some variables 7 represents valid code, decided to exclude it from the sentinel code cleaning

In [ ]:
## Check APResInj ( to see if it removed the 7 value)
## AFScorRb & AFScorDs, 88 is a valid value
## AEducLvl, 8 is valid value
## ANCatAdm ANCatRhb ANCatDis, 8 is a valid value

col_to_leave = ["AFScorRb", "AFScorDs", "AEducLvl", "ANCatAdm", "ANCatRhb", "ANCatDis"]

In [ ]:
def clean_column(series):
    codes = sentinel_codes(series)
    return series.replace(list(codes.values()), pd.NA)

form1_cols = f1.columns.tolist()

f1_clean = f1.copy()

for col in form1_cols:
    if col in col_to_leave:
        continue
    if pd.api.types.is_numeric_dtype(f1[col]):
        codes = sentinel_codes(f1[col])
        f1_clean[col] = f1[col].replace(list(codes.values()), pd.NA)

In [ ]:
FIXED_DECIMAL_SENTINELS = [777.77, 888.88, 999.99, 999.9]

# Only includes columns I've actually confirmed carry this pattern
confirmed_decimal_sentinel_cols = [
    "ARsTScr", "ARsStEr"
    # add others here once verified via the check above
]

for col in confirmed_decimal_sentinel_cols:
    f1_clean[col] = f1_clean[col].replace(FIXED_DECIMAL_SENTINELS, np.nan)

In [ ]:
# AExtCsIj is categorical (ICD-10 external-cause-of-injury codes), not continuous.
# 999.999 is the "unknown cause" sentinel - recode explicitly rather than to NaN,
# since "Unknown cause" and "not collected" are different things worth keeping separate.

f1_clean["AExtCsIj"] = f1_clean["AExtCsIj"].astype(str).str.strip()
f1_clean["AExtCsIj"] = f1_clean["AExtCsIj"].replace(
    {"999.999": "Unknown"}
)

# Confirm the recode landed and see what's left
f1_clean["AExtCsIj"].value_counts(dropna=False).head(20)

I repeated the same method for form 2 and records datasets.

In form 2, these are the columns that have weird setinel values:

confirmed_decimal_sentinel_cols = [
    "BRsTScr", "BRsStEr",
]

## Anuoluwa Data joining and inclusion criteria

In [ ]:
# Form I + Record Status: 1:1 merge on UniID
baseline = f1_clean.merge(rec_clean, on="UniID", how="left", validate="1:1")

# Attach baseline info to every Form II row for that person (many-to-one)
full = f2_clean.merge(baseline, on="UniID", how="left", validate="m:1")

In [ ]:
# ---------------------------------------------------------------
# Step 4: Inclusion criteria — exclusion code definitions
# ---------------------------------------------------------------
APRESDIS_EXCLUDE = [7, 8, 99]          # Deceased, Other/unclassified, Unknown
APRESDIS_ASSISTED_LIVING_CODE = 10      # included by default — toggle for sensitivity check
BPLCRES_EXCLUDE = [8, 99]               # Other/unclassified, Unknown
BRHSPNBR_EXCLUDE = [9]                  # Unknown
BRHSPNBR_COUNT_UNKNOWN_CODE = 8         # rehospitalized, count unknown — kept as valid flag

# ---------------------------------------------------------------
# Step 5: Row-level eligibility flags
# ---------------------------------------------------------------
full["valid_discharge_loc"] = ~full["APResDis"].isin(APRESDIS_EXCLUDE)
full["is_assisted_living_discharge"] = full["APResDis"] == APRESDIS_ASSISTED_LIVING_CODE

blank_bad_living = full["BPlcRes"].isna() & (~full["is_lost_to_followup"])
full["valid_living_loc"] = (~full["BPlcRes"].isin(BPLCRES_EXCLUDE)) & (~blank_bad_living)
full["unexpected_blank_living_loc"] = blank_bad_living

blank_bad_rehosp = full["BRhspNbr"].isna() & (~full["is_lost_to_followup"])
full["valid_rehosp_status"] = (~full["BRhspNbr"].isin(BRHSPNBR_EXCLUDE)) & (~blank_bad_rehosp)
full["rehosp_count_unknown"] = full["BRhspNbr"] == BRHSPNBR_COUNT_UNKNOWN_CODE
full["unexpected_blank_rehosp"] = blank_bad_rehosp

# ---------------------------------------------------------------
# Step 6: Row-level construct validity — pooled across variables
# ---------------------------------------------------------------
sdoh_vars = ["BPlcRes", "BMarStat", "BEducLvl", "BPrLvlSt", "BFmIncLv", "BPrimPay", "BMdTrnsp"]

# NOTE: PHQ (BBPHQMDS, BBPHQSDS) folded into functional ability per user decision,
# alongside self-care (BSFSCRS) and physical independence (BCHPITot).
# "Pooled" means: a year counts as a valid timepoint if ANY one of the listed
# variables is non-missing that year — it does not require the same specific
# variable to repeat across years.
functional_vars = ["BSFSCRS", "BCHPITot", "BBPHQMDS", "BBPHQSDS"]

full["sdoh_row_valid"] = full[sdoh_vars].notna().any(axis=1) & full["valid_living_loc"]
full["functional_row_valid"] = full[functional_vars].notna().any(axis=1)

# ---------------------------------------------------------------
# Step 7: Collapse to person-level eligibility
# ---------------------------------------------------------------
person_summary = full.groupby("UniID").agg(
    n_sdoh_timepoints=("sdoh_row_valid", "sum"),
    n_functional_timepoints=("functional_row_valid", "sum"),
    baseline_discharge_valid=("valid_discharge_loc", "first"),
    has_valid_living_loc=("valid_living_loc", "any"),
    has_valid_rehosp=("valid_rehosp_status", "any"),
).reset_index()

person_summary["meets_sdoh_criterion"] = person_summary["n_sdoh_timepoints"] >= 2
person_summary["meets_functional_criterion"] = person_summary["n_functional_timepoints"] >= 2
person_summary["eligible"] = (
    person_summary["baseline_discharge_valid"]
    & person_summary["has_valid_living_loc"]
    & person_summary["has_valid_rehosp"]
    & person_summary["meets_sdoh_criterion"]
    & person_summary["meets_functional_criterion"]
)

# ---------------------------------------------------------------
# Step 8: Transparency funnel
# ---------------------------------------------------------------
funnel = pd.Series({
    "Total unique people (Form I)": f1_clean["UniID"].nunique(),
    "Valid discharge location": person_summary["baseline_discharge_valid"].sum(),
    "+ valid living location & rehosp status": (
        person_summary["baseline_discharge_valid"]
        & person_summary["has_valid_living_loc"]
        & person_summary["has_valid_rehosp"]
    ).sum(),
    "+ 2 or more SDOH timepoints": (
        person_summary["baseline_discharge_valid"]
        & person_summary["has_valid_living_loc"]
        & person_summary["has_valid_rehosp"]
        & person_summary["meets_sdoh_criterion"]
    ).sum(),
    "+ 2 or more functional timepoints (final eligible)": person_summary["eligible"].sum(),
})
print(funnel)
print("\nUnexpected blank living location rows:", full["unexpected_blank_living_loc"].sum())
print("Unexpected blank rehosp rows:", full["unexpected_blank_rehosp"].sum())

# ---------------------------------------------------------------
# Step 9: Final analysis-ready cohort
# ---------------------------------------------------------------
eligible_ids = person_summary.loc[person_summary["eligible"], "UniID"]
analysis_df = full[full["UniID"].isin(eligible_ids)].copy()

print(f"\nFinal analysis cohort: {analysis_df['UniID'].nunique()} people, {len(analysis_df)} person-year rows")

**NOTES**

- Pooled timepoints: a person meets the "2+ SDOH timepoints" bar if any 2 different years each have at least one SDOH variable populated — not necessarily the same variable across those years. Same logic for functional ability.
- PHQ → functional ability: BBPHQMDS (Major Depressive Syndrome) and BBPHQSDS (Severity of Depression Score) are now pooled into functional_vars alongside self-care and physical independence, per your call.
- One side effect of pooling: because PHQ items only exist from Oct-2016/Oct-2000 onward and CHART goes back to 1995, a person could satisfy the functional-ability criterion using two completely different instruments in two different eras (e.g., CHART in 1998 + PHQ in 2018)

Total unique people (Form I)                          35675
Valid discharge location                              32514
+ +valid living location & rehosp status               32104
+ +2 or more SDOH timepoints                           17927
+ +2 or more functional timepoints (final eligible)    17927
dtype: int64

Unexpected blank living location rows: 0
Unexpected blank rehosp rows: 0

Final analysis cohort: 17927 people, 88023 person-year rows

## Anuoluwa Possible outcome indicators:
The three outcome indicators

a. Returned home after rehab

- APResDis — Place of Residence at Discharge (Form I). This is the baseline discharge-disposition outcome (home vs. institutional settings).
- BPlcRes — Place of Residence (Form II, per annual follow-up) — lets you track whether "returned home" status is sustained or changes across post-injury years, not just at discharge.
- APResInj — Place of Residence at Injury (Form I) is a useful covariate/comparator (were they already living somewhere other than a private home pre-injury?).

b. Rehospitalized
All on Form II, since rehospitalization is a post-discharge/follow-up construct:

- BRhspNbr — Number of Rehospitalizations in last 12 months (your binary/count outcome)
- BRhspDaT — Total Days Rehospitalized in last 12 months
- BRhspDa1–BRhspDa8 — Days rehospitalized, per episode (up to 8 episodes)
- BRhspRs1–BRhspRs8 — Reason for rehospitalization, per episode (this answers the "most common reasons" sub-question)
- BYear — post-injury year, needed to compute "% rehospitalized at each post-injury year"

c. Unable to drive

- BDrvModV — Modified Vehicle, Driving (Form II) — this is the direct driving-ability item.
- BModVhcl — Modified Vehicle, Type (context: what kind of modification, if any)
- BMdTrnsp — Mode of Transportation (broader participation context — useful covariate/secondary outcome, since someone may not drive but still travel independently)

## Record Status (rec_public_2021) Cleaning


In [ ]:
# Record Status (rec_public_2021) Cleaning



# Make sure rec is loaded
rec_clean = rec.copy()


# Clean Calendar Year Columns (Convert float YYYY.0 -> nullable integer Int64)
rec_clean["vital_status_year"] = rec_clean["SVitDate"].astype("Int64")
rec_clean["death_year"] = rec_clean["SDthDt"].astype("Int64")


# Make Standardized Tracking & Vital Status Flags
# SStatCur is the official hierarchical tracking status

rec_clean["SStatCur"] = rec_clean["SStatCur"].astype(str).str.strip()

# Primary mortality indicator (1 = confirmed deceased, 0 = alive/censored)
rec_clean["is_deceased"] = (rec_clean["SStatCur"] == "Deceased").astype(int)

# Active surveillance
rec_clean["is_active_eligible"] = (rec_clean["SStatCur"] == "Eligible").astype(int)

# Attrition flags
# is_lost_to_followup feeds Step 5 in the team's eligibility check
rec_clean["is_lost_to_followup"] = (rec_clean["SStatCur"] == "Lost").astype(int)
rec_clean["is_withdrew_or_untraced"] = rec_clean["SStatCur"].isin(
    ["Withdrew Consent", "ID Unknown"]
).astype(int)

# Clinical recovery (exempt from community follow-up)
rec_clean["is_neuro_recovered"] = rec_clean["SStatCur"].isin(
    ["Norm-Min", "Minimal", "Normal"]
).astype(int)


In [ ]:
# Clean Cause of Death Variables (SCsDth1 - SCsDth5)
# In SCsDth1:
#   '888.8' = Patient Alive (recode to NaN for cause-of-death analyses)
#   '999.9' = Deceased, cause unknown (preserve as "Unknown")

cod_cols = ["SCsDth1", "SCsDth2", "SCsDth3", "SCsDth4", "SCsDth5"]

for col in cod_cols:
    rec_clean[col] = rec_clean[col].astype(str).str.strip()
    rec_clean[col] = rec_clean[col].replace(
        {"888.8": np.nan, "nan": np.nan, "None": np.nan, "": np.nan}
    )


In [ ]:
# Clinical Cause-of-Death Categorization (for Community Integration Impact)
# Maps ICD-10 primary causes into organ-system categories

def categorize_cause_of_death(code):
    if pd.isna(code):
        return np.nan
    c = str(code).upper()
    if c == "999.9":
        return "Unknown / Unspecified"
    elif c.startswith(("A40", "A41")) or c == "B37.7":
        return "Sepsis / Severe Infection"
    elif c.startswith(("J13", "J15", "J18", "J69")):
        return "Pneumonia / Respiratory Infection"
    elif c.startswith(("J96", "J80", "R09.2")):
        return "Acute Respiratory Failure"
    elif c.startswith(("I21", "I25", "I46", "I50")):
        return "Cardiovascular Disease"
    elif c.startswith("I26"):
        return "Pulmonary Embolism"
    elif c.startswith(("I60", "I61", "I64")):
        return "Stroke / Cerebrovascular"
    elif c.startswith(("C", "D")):
        return "Malignant Neoplasms"
    elif c.startswith(("V", "W", "X", "Y")):
        return "External Trauma / Accidents / Suicide"
    else:
        return "Other Medical Conditions"

rec_clean["primary_death_category"] = rec_clean["SCsDth1"].apply(categorize_cause_of_death)

# Confirm cleaned record status matches Form I 1:1
print(f"Record Status cleaned: {len(rec_clean):,} rows, {rec_clean['UniID'].nunique():,} unique patients")
print(f"Confirmed Deceased: {rec_clean['is_deceased'].sum():,} | Active Eligible: {rec_clean['is_active_eligible'].sum():,}")